import pandas as pd 
import numpy as np

# Load the GlobalMart dataset from Excel (two sheets we need: products + transactions)
product_df = pd.read_excel(r'https://cdn.enqurious.com/others/2adcae84-34ab-4421-a4aa-5580392d32ac_globalmart.xlsx', sheet_name='products')
trans_df   = pd.read_excel(r'https://cdn.enqurious.com/others/2adcae84-34ab-4421-a4aa-5580392d32ac_globalmart.xlsx', sheet_name='transactions')

# Merge: bring product info (name, category, sub_category) into each transaction row
# SQL equivalent: SELECT t.*, p.product_name, p.category ...
#                 FROM transactions t JOIN products p ON t.product_id = p.product_id
df = trans_df.merge(product_df, on='product_id')

# Check what columns we have after the merge
df.columns

In [1]:
import pandas as pd 
import numpy as np

# Load datasets
product_df = pd.read_excel(r'https://cdn.enqurious.com/others/2adcae84-34ab-4421-a4aa-5580392d32ac_globalmart.xlsx', sheet_name='products')
trans_df = pd.read_excel(r'https://cdn.enqurious.com/others/2adcae84-34ab-4421-a4aa-5580392d32ac_globalmart.xlsx', sheet_name='transactions')

df = trans_df .merge(product_df, on='product_id')
df.columns

Index(['id', 'order_id', 'product_id', 'sales_amt', 'qty', 'discount',
       'profit_amt', 'product_name', 'colors', 'category', 'sub_category',
       'date_added', 'manufacturer', 'sizes', 'upc', 'weight',
       'product_photos_qty', 'Unnamed: 11'],
      dtype='object')

# Keep only the columns needed for this analysis
df = df[['product_id', 'sales_amt', 'qty', 'discount', 'product_name', 'category', 'sub_category']]

# Compute the unit price from sales_amt / qty
# WHY np.where? Guard against qty=0 (division by zero) → return NaN instead of crashing
# SQL equivalent: CASE WHEN qty = 0 THEN NULL ELSE sales_amt / qty END AS price
df['price'] = np.where(df['qty'] == 0, np.nan, df['sales_amt'] / df['qty'])

# Apply the seasonal discount to get the post-discount price
# IMPORTANT: discount is stored as a DECIMAL fraction (0.45 = 45% off)
# Do NOT divide by 100 — the formula is simply: price * (1 - discount)
# SQL equivalent: price * (1 - discount) AS discounted_price
df['discounted_price'] = df['price'] * (1 - df['discount'])

# Classify each product into a price tier based on its discounted price
#
# SQL equivalent:
#   CASE WHEN discounted_price > 200 THEN 'Premium'
#        WHEN discounted_price > 100 THEN 'Mid-range'
#        ELSE 'Budget'
#   END AS price_category
#
# Python list comprehension: ['Premium' if ... else 'Mid-range' if ... else 'Budget' for p in col]
# Conditions are evaluated left to right — the first True condition wins (same as CASE WHEN)
df['price_category'] = ['Premium' if price > 200 else 'Mid-range' if price > 100 else 'Budget' for price in df['discounted_price']]

In [2]:
df.head()

,id,order_id,product_id,sales_amt,qty,discount,profit_amt,product_name,colors,category,sub_category,date_added,manufacturer,sizes,upc,weight,product_photos_qty,Unnamed: 11
0,2698,CA-2014-145317,FUR-BO-10001798,261.9600,2,0.00,41.9136,Bush Somerset Collection Bookcase,Pink,Furniture,Bookcases,2016-04-01,NaN,9,640000000000,NaN,4,NaN
1,6827,CA-2016-118689,FUR-CH-10000454,731.9400,3,0.00,219.5820,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Pink,Furniture,Chairs,2016-11-04,NaN,"10,7,6,9,8",664000000000,NaN,2,NaN
2,8154,CA-2017-140151,OFF-LA-10000240,14.6200,2,0.00,6.8714,Self-Adhesive Address Labels for Typewriters b...,Pink,Office Supplies,Labels,2016-08-01,NaN,11.5,887000000000,NaN,3,NaN
3,2624,CA-2017-127180,FUR-TA-10000577,957.5775,5,0.45,-383.0310,Bretford CR4500 Series Slim Rectangular Table,Blue,Furniture,Tables,2016-11-15,NaN,"10.5,10,8.5,8,13",888000000000,NaN,0,NaN
4,4191,CA-2017-166709,OFF-ST-10000760,22.3680,2,0.20,2.5164,Eldon Fold 'N Roll Cart System,Blue,Office Supplies,Storage,2016-08-01,NaN,NaN,604000000000,NaN,0,NaN


In [3]:
df= df[['product_id', 'sales_amt', 'qty', 'discount','product_name', 'category', 'sub_category']]

# Calculate the price
df['price'] = np.where(df['qty'] == 0, np.nan, df['sales_amt'] / df['qty'])

# Calculate the discounted price
df['discounted_price'] = df['price'] * (1 - df['discount'])

# Build the category-level discount impact summary
#
# SQL equivalent:
#   SELECT category,
#     AVG(sales_amt) AS original_price,
#     AVG(discounted_price) AS discounted_price
#   FROM df
#   GROUP BY category;
#
# Named aggregation syntax: agg(output_col_name=('source_col', 'agg_function'))
# This is the cleanest way to rename the aggregated columns in one step
category_summary = df.groupby('category').agg(
    original_price   = ('sales_amt',        'mean'),
    discounted_price = ('discounted_price', 'mean')
).reset_index()

In [4]:
# Categorize products based on their discounted prices
df['price_category'] = ['Premium' if price > 200 else 'Mid-range' if price > 100 else 'Budget' for price in df['discounted_price']]

In [5]:
df.head()

,product_id,sales_amt,qty,discount,product_name,category,sub_category,price,discounted_price,price_category
0,FUR-BO-10001798,261.9600,2,0.00,Bush Somerset Collection Bookcase,Furniture,Bookcases,130.9800,130.980000,Mid-range
1,FUR-CH-10000454,731.9400,3,0.00,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs,243.9800,243.980000,Premium
2,OFF-LA-10000240,14.6200,2,0.00,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels,7.3100,7.310000,Budget
3,FUR-TA-10000577,957.5775,5,0.45,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables,191.5155,105.333525,Mid-range
4,OFF-ST-10000760,22.3680,2,0.20,Eldon Fold 'N Roll Cart System,Office Supplies,Storage,11.1840,8.947200,Budget


In [6]:
df[(df['price_category'] == 'Premium') & (df['category'] == 'Office Supplies')]

,product_id,sales_amt,qty,discount,product_name,category,sub_category,price,discounted_price,price_category
79,OFF-AP-10002118,208.160,1,0.0,"1.7 Cubic Foot Compact ""Cube"" Office Refrigera...",Office Supplies,Appliances,208.160,208.1600,Premium
144,OFF-AP-10001058,839.430,3,0.0,Sanyo 2.5 Cubic Foot Mid-Size Office Refrigera...,Office Supplies,Appliances,279.810,279.8100,Premium
247,OFF-AP-10002945,1503.250,5,0.0,Honeywell Enviracaire Portable HEPA Air Cleane...,Office Supplies,Appliances,300.650,300.6500,Premium
353,OFF-BI-10004995,4355.168,4,0.2,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,1088.792,871.0336,Premium
454,OFF-ST-10000142,665.408,2,0.2,Deluxe Rollaway Locking File with Drawer,Office Supplies,Storage,332.704,266.1632,Premium
...,...,...,...,...,...,...,...,...,...,...
9446,OFF-BI-10003527,2033.584,2,0.2,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,1016.792,813.4336,Premium
9563,OFF-BI-10001359,896.990,1,0.0,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,896.990,896.9900,Premium
9609,OFF-BI-10004390,673.568,2,0.2,GBC DocuBind 200 Manual Binding Machine,Office Supplies,Binders,336.784,269.4272,Premium
9774,OFF-AP-10002945,2405.200,8,0.0,Honeywell Enviracaire Portable HEPA Air Cleane...,Office Supplies,Appliances,300.650,300.6500,Premium


In [7]:
# Group by category and calculate the average original and discounted prices
category_summary = df.groupby('category').agg(
    original_price=('sales_amt', 'mean'),
    discounted_price=('discounted_price', 'mean')
).reset_index()

In [8]:
category_summary 

,category,original_price,discounted_price
0,Furniture,349.561855,75.305466
1,Office Supplies,120.649472,29.131460
2,Technology,444.365035,100.792155
